# Mini-Projeto — Análise Exploratória de Dados: Base Varejo

**Aluna:** Aline Góes Dos Santos
**Turma:** T8 — Análise de Dados com Python (SCTEC)
**Módulo 1 — Semana 07**

**Objetivo:** transformar a base bruta de vendas de um varejo em informação útil para o negócio — diagnosticar a qualidade dos dados, limpar apenas o que for comprovadamente problemático (com justificativa), analisar clientes, compras e categorias e exportar uma base tratada (`varejo_limpo.csv`).

**Sequência do notebook:** Ler → Diagnosticar → Limpar (funções) → Analisar → Exportar → Comunicar.

## Sprint 1 — Leitura e diagnóstico

### 1.1 Leitura do CSV
A base original vem compactada (`Base_Varejo.csv.zip`, com o arquivo `Base Varejo.csv` dentro), na mesma pasta do notebook. O `pd.read_csv()` lê o `.zip` diretamente.
Ao abrir o arquivo num editor de texto, vemos que o separador é **ponto e vírgula (`;`)**, por isso usamos `sep=";"`.

> **No Google Colab:** envie o arquivo `Base_Varejo.csv.zip` pelo ícone de pasta (menu lateral) antes de executar. Não é preciso descompactar.

In [1]:
import pandas as pd

# Caminho da base original: o .zip fica na mesma pasta do notebook (no Colab, enviado pelo ícone de pasta)
CAMINHO_BASE = "Base_Varejo.csv.zip"

# LEITURA: separador ";" (verificado no arquivo bruto)
df = pd.read_csv(CAMINHO_BASE, sep=";")

# Primeiras linhas
df.head()

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN


In [2]:
# Estrutura: quantidade de registros (linhas) e colunas
print("Registros (linhas):", df.shape[0])
print("Colunas:", df.shape[1])
print("\nNomes das colunas:", list(df.columns))

Registros (linhas): 830000
Colunas: 14

Nomes das colunas: ['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']


In [3]:
# Tipos de dados de cada coluna
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  str    
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  str    
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  str    
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  str    
 9   PR_NOME      830000 non-null  str    
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), str(5)
memory usage: 88.7 MB


### 1.2 O que cada coluna significa
Dicionário montado a partir dos nomes das colunas e da conferência dos próprios dados (a base não traz dicionário oficial):

| Coluna | Significado | Tipo de informação |
|---|---|---|
| `DATA` | Data da compra (formato dia/mês/ano) | Data |
| `CO_ID` | Identificador da **compra** (cupom) | Identificador — não é medida |
| `CL_ID` | Identificador do **cliente** | Identificador — não é medida |
| `CL_GENERO` | Gênero do cliente (F / M) | Categoria |
| `CL_EC` | Estado civil do cliente, codificado de 1 a 5 | Categoria codificada — não é medida |
| `CL_FHL` | Número de filhos do cliente (0 a 4) | **Numérica com significado** |
| `CL_SEG` | Segmento do cliente (A / B / C) | Categoria |
| `PR_ID` | Identificador do produto | Identificador — não é medida |
| `PR_CAT` | Categoria do produto | Categoria |
| `PR_NOME` | Nome do produto | Texto |
| `Unnamed: 10` a `Unnamed: 13` | Colunas criadas pelos `;;;;` que sobram no fim de cada linha do arquivo | Sem conteúdo (verificado abaixo) |

### 1.3 Unidade da análise: o que uma linha representa?
Antes de contar qualquer coisa, é preciso saber se cada linha é um cliente, uma compra ou um item. Verificamos com os identificadores:

In [4]:
print("Linhas na base:              ", len(df))
print("Compras distintas (CO_ID):   ", df["CO_ID"].nunique())
print("Clientes distintos (CL_ID):  ", df["CL_ID"].nunique())
print("Produtos distintos (PR_ID):  ", df["PR_ID"].nunique())

# Cada compra pertence a um único cliente e a uma única data?
print("\nMáx. de clientes por compra:", df.groupby("CO_ID")["CL_ID"].nunique().max())
print("Máx. de datas por compra:   ", df.groupby("CO_ID")["DATA"].nunique().max())

# Os dados do cliente são constantes para o mesmo CL_ID? (1 = sempre o mesmo valor)
print("\nMáx. de valores diferentes por cliente:")
print(df.groupby("CL_ID")[["CL_GENERO", "CL_EC", "CL_FHL", "CL_SEG"]].nunique().max())

Linhas na base:               830000
Compras distintas (CO_ID):    18471
Clientes distintos (CL_ID):   1000
Produtos distintos (PR_ID):   229

Máx. de clientes por compra: 1
Máx. de datas por compra:    1

Máx. de valores diferentes por cliente:


CL_GENERO    1
CL_EC        1
CL_FHL       1
CL_SEG       1
dtype: int64


**Conclusão sobre a unidade:** cada **linha é um item (produto) dentro de uma compra**.
- 830.000 linhas = 830.000 itens registrados, distribuídos em **18.471 compras** feitas por **1.000 clientes**.
- Cada compra tem um único cliente e uma única data; os atributos do cliente (gênero, estado civil, filhos, segmento) se repetem em todas as linhas dele e nunca mudam.
- Portanto: **contar linhas = contar itens**, não clientes nem compras. Para falar de clientes ou compras é preciso usar `nunique()`.
- A base **não tem coluna de valor (R$) nem de quantidade**, então não é possível calcular faturamento.

### 1.4 Qualidade dos dados

#### a) Valores ausentes por coluna (quantidade e %)

In [5]:
ausentes = pd.DataFrame({
    "ausentes": df.isna().sum(),
    "percentual_%": (df.isna().mean() * 100).round(2),
})
ausentes

,ausentes,percentual_%
DATA,0,0.0
CO_ID,0,0.0
CL_ID,0,0.0
CL_GENERO,0,0.0
CL_EC,0,0.0
CL_FHL,0,0.0
CL_SEG,0,0.0
PR_ID,0,0.0
PR_CAT,0,0.0
PR_NOME,0,0.0


As 10 colunas reais têm **0 valores ausentes**. As 4 colunas `Unnamed` estão **100% vazias** — são sobras do separador no fim das linhas, não informação.

#### b) Textos vazios ou compostos apenas por espaços
O `isna()` não detecta texto vazio (`""`) nem texto só com espaços, então verificamos separadamente — também espaços sobrando no início/fim.

In [6]:
colunas_texto = ["DATA", "CL_GENERO", "CL_SEG", "PR_CAT", "PR_NOME"]

verif_texto = pd.DataFrame({
    "vazio_ou_so_espacos": [df[c].astype("string").str.strip().eq("").sum() for c in colunas_texto],
    "espacos_nas_bordas":  [(df[c].astype("string") != df[c].astype("string").str.strip()).sum() for c in colunas_texto],
    "com_minusculas":      [(df[c].astype("string") != df[c].astype("string").str.upper()).sum() for c in colunas_texto],
}, index=colunas_texto)
verif_texto

,vazio_ou_so_espacos,espacos_nas_bordas,com_minusculas
DATA,0,0,0
CL_GENERO,0,0,0
CL_SEG,0,0,0
PR_CAT,0,0,0
PR_NOME,0,0,7320


#### c) Inconsistências de texto e marcadores de ausência disfarçados
Conferimos os valores de cada categoria (com `dropna=False`, para não esconder nada):

In [7]:
print(df["CL_GENERO"].value_counts(dropna=False), "\n")
print(df["CL_SEG"].value_counts(dropna=False), "\n")
print(df["PR_CAT"].value_counts(dropna=False))

CL_GENERO
F    432576
M    397424
Name: count, dtype: int64 

CL_SEG
B    530163
C    232101
A     67736
Name: count, dtype: int64 

PR_CAT
ALIMENTOS     434767
HIGIENE       155574
LIMPEZA       145754
BEBIDAS        43299
PET            32399
ACESSORIOS     14557
#N/D            3650
Name: count, dtype: int64


In [8]:
# "#N/D" é o erro de PROCV do Excel: na prática é um valor AUSENTE escrito como texto
nd = df[(df["PR_CAT"] == "#N/D") | (df["PR_NOME"] == "#N/D")]
print("Linhas com '#N/D':", len(nd))
print("PR_ID dessas linhas:", nd["PR_ID"].unique())
nd.head()

Linhas com '#N/D': 3650
PR_ID dessas linhas: [107]


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
82,01/02/2019,1078,290,F,1,0,B,107,#N/D,#N/D,NaN,NaN,NaN,NaN
223,01/02/2019,1103,957,M,2,3,B,107,#N/D,#N/D,NaN,NaN,NaN,NaN
640,01/02/2019,1482,453,F,1,0,B,107,#N/D,#N/D,NaN,NaN,NaN,NaN
857,01/02/2019,1683,438,F,1,0,B,107,#N/D,#N/D,NaN,NaN,NaN,NaN
917,01/02/2019,1793,608,F,3,0,C,107,#N/D,#N/D,NaN,NaN,NaN,NaN


In [9]:
# Nome de produto escrito com letra minúscula no meio (inconsistência de digitação)
minusc = df[df["PR_NOME"] != df["PR_NOME"].str.upper()]
print(minusc["PR_NOME"].value_counts())

PR_NOME
REFRIGERANTE LIMaO    7320
Name: count, dtype: int64


In [10]:
# Mesmo nome de produto com mais de um código (PR_ID)
ids_por_nome = df.groupby("PR_NOME")["PR_ID"].nunique()
print("Nomes de produto distintos:", df["PR_NOME"].nunique())
print("Nomes com mais de 1 PR_ID:", (ids_por_nome > 1).sum())
df[df["PR_NOME"].isin(["ARROZ", "BANANA"])].groupby(["PR_NOME", "PR_ID", "PR_CAT"]).size()

Nomes de produto distintos: 118
Nomes com mais de 1 PR_ID: 109


PR_NOME  PR_ID  PR_CAT   
ARROZ    8      ALIMENTOS    3624
         227    ALIMENTOS    3594
BANANA   13     ALIMENTOS    3667
         221    ALIMENTOS    3718
dtype: int64

#### d) Datas
A coluna `DATA` foi lida como texto. Conferimos se todas seguem o formato dia/mês/ano:

In [11]:
datas = pd.to_datetime(df["DATA"], format="%d/%m/%Y", errors="coerce")
print("Datas inválidas:", datas.isna().sum())
print("Período:", datas.min().date(), "a", datas.max().date())

Datas inválidas: 0
Período: 2019-01-04 a 2022-12-08


> Observação: um `df["DATA"].min()` direto no texto daria o resultado errado (compara "01/01/2020" com "04/01/2019" letra a letra). Por isso a conversão para data é necessária.

#### e) Linhas duplicadas

In [12]:
dup = df.duplicated()          # True a partir da 2ª ocorrência de uma linha idêntica
print("Linhas duplicadas (idênticas em todas as colunas):", dup.sum())
print("Percentual da base:", round(dup.mean() * 100, 2), "%")

# Visualizar ANTES de decidir: exemplo de uma mesma compra
exemplo = df[df.duplicated(keep=False) & (df["CO_ID"] == 1000)].sort_values("PR_NOME")
exemplo[["DATA", "CO_ID", "CL_ID", "PR_ID", "PR_NOME"]].head(8)

Linhas duplicadas (idênticas em todas as colunas): 96553
Percentual da base: 11.63 %


,DATA,CO_ID,CL_ID,PR_ID,PR_NOME
3,01/02/2019,1000,534,4,ABACAXI
40,01/02/2019,1000,534,4,ABACAXI
34,01/02/2019,1000,534,225,ATUM
51,01/02/2019,1000,534,225,ATUM
7,01/02/2019,1000,534,11,AZEITE
46,01/02/2019,1000,534,11,AZEITE
14,01/02/2019,1000,534,13,BANANA
19,01/02/2019,1000,534,13,BANANA


In [13]:
# Quantas vezes as linhas repetidas aparecem? E elas estão sempre dentro da MESMA compra?
repeticoes = df[df.duplicated(keep=False)].groupby(list(df.columns), dropna=False).size()
print("Distribuição: nº de vezes que a mesma linha aparece")
print(repeticoes.value_counts().sort_index())
print("\nCompras com pelo menos um item repetido:", df.loc[dup, "CO_ID"].nunique(), "de", df["CO_ID"].nunique())

Distribuição: nº de vezes que a mesma linha aparece
2    79608
3     7601
4      532
5       33
6        3
Name: count, dtype: int64

Compras com pelo menos um item repetido: 14566 de 18471


**Interpretação das duplicatas:** as 96.553 linhas repetidas são **o mesmo produto, na mesma compra, do mesmo cliente, no mesmo dia**. Como a base **não tem coluna de quantidade** nem número sequencial do item no cupom, uma repetição é exatamente como o sistema registraria o cliente levando **2 unidades** do mesmo produto (ex.: duas bananas, dois refrigerantes). Não há nenhum campo que permita **confirmar** que é erro de registro.

#### Resumo do diagnóstico (Sprint 1)

| Verificação | Resultado |
|---|---|
| Tamanho | 830.000 registros × 14 colunas (10 com dados + 4 vazias) |
| Unidade | 1 linha = 1 item de uma compra · 18.471 compras · 1.000 clientes |
| Valores ausentes (nulos) | 0 nas 10 colunas reais · 830.000 (100%) em cada uma das 4 colunas `Unnamed` |
| Textos vazios / só espaços | 0 em todas as colunas de texto |
| Espaços nas bordas | 0 |
| Ausência disfarçada `#N/D` | 3.650 linhas (`PR_CAT` e `PR_NOME`, todas do `PR_ID` 107) |
| Inconsistência de grafia | 7.320 linhas com `REFRIGERANTE LIMaO` (minúscula no meio) |
| Mesmo produto com 2 códigos | 109 dos 118 nomes têm 2 `PR_ID` — **não alterar sem investigar** |
| Datas | 0 inválidas; vieram como texto; período 04/01/2019 a 08/12/2022 |
| Duplicatas | 96.553 linhas idênticas (11,63%), todas dentro da mesma compra — **não confirmadas como erro** |

## Sprints 2 e 3 — Funções, limpeza e padronização

### 2.1 Funções reutilizáveis
Todas recebem **parâmetro** e devolvem um **resultado com `return`** (não apenas imprimem):

| Função | Parâmetro | Retorno | Papel |
|---|---|---|---|
| `padronizar_texto(serie)` | uma coluna de texto | a coluna padronizada | tratamento de texto |
| `tratar_base(df_bruto, remover_duplicatas)` | a base bruta | a base tratada | **função de tratamento** |
| `diagnostico_qualidade(base)` | uma base | tabela com os indicadores | compara antes × depois |
| `resumo_numerico(serie)` | uma coluna numérica | dicionário com contagem, média, mín. e máx. | **função de resumo** |

In [14]:
def padronizar_texto(serie):
    """Remove espaços nas bordas e deixa o texto em MAIÚSCULAS.
    Unifica representações equivalentes ('Limão', 'LIMaO ', 'LIMÃO') sem mudar o significado."""
    return serie.astype("string").str.strip().str.upper()


def tratar_base(df_bruto, remover_duplicatas=False):
    """Recebe a base bruta e devolve uma CÓPIA tratada (a original não é alterada).
    remover_duplicatas=False: as linhas repetidas são mantidas (ver justificativa no notebook)."""
    base = df_bruto.copy()

    # 1) Remove apenas colunas 100% vazias (sobras do ';' no fim das linhas)
    colunas_vazias = [c for c in base.columns if base[c].isna().all()]
    base = base.drop(columns=colunas_vazias)

    # 2) Padroniza as colunas textuais
    for coluna in ["CL_GENERO", "CL_SEG", "PR_CAT", "PR_NOME"]:
        base[coluna] = padronizar_texto(base[coluna])

    # 3) '#N/D' (erro de PROCV) = produto não identificado -> rótulo explícito, sem inventar produto
    for coluna in ["PR_CAT", "PR_NOME"]:
        base[coluna] = base[coluna].replace("#N/D", "NAO IDENTIFICADO")

    # 4) Converte a data de texto (dd/mm/aaaa) para o tipo data
    base["DATA"] = pd.to_datetime(base["DATA"], format="%d/%m/%Y")

    # 5) Duplicatas: só remove se o parâmetro pedir
    if remover_duplicatas:
        base = base.drop_duplicates()

    return base


def diagnostico_qualidade(base):
    """Devolve uma tabela (Series) com os principais indicadores de qualidade da base."""
    textos = base.select_dtypes(exclude=["number", "datetime"]).columns
    return pd.Series({
        "registros": len(base),
        "colunas": base.shape[1],
        "colunas 100% vazias": int(base.isna().all().sum()),
        "valores ausentes (total)": int(base.isna().sum().sum()),
        "textos vazios/só espaços": int(sum(base[c].astype("string").str.strip().eq("").sum() for c in textos)),
        "linhas com '#N/D'": int(base["PR_CAT"].astype("string").eq("#N/D").sum()),
        "linhas com minúscula em PR_NOME": int((base["PR_NOME"].astype("string") != base["PR_NOME"].astype("string").str.upper()).sum()),
        "nomes de produto distintos": base["PR_NOME"].nunique(),
        "linhas duplicadas": int(base.duplicated().sum()),
        "DATA é do tipo data?": pd.api.types.is_datetime64_any_dtype(base["DATA"]),
    })


def resumo_numerico(serie):
    """Devolve as 4 estatísticas obrigatórias de uma coluna numérica."""
    return {
        "contagem_validos": int(serie.count()),   # count() ignora ausentes
        "media": round(float(serie.mean()), 2),
        "minimo": int(serie.min()),
        "maximo": int(serie.max()),
    }

### 2.2 Decisões de limpeza (problema → decisão → motivo)

| Problema encontrado | Decisão | Motivo |
|---|---|---|
| 4 colunas `Unnamed` 100% vazias | **Remover as colunas** | Não têm nenhum dado; são sobras do `;` no fim das linhas. |
| Nenhum nulo nas colunas reais | **Manter** (nada a fazer) | Verificar que não há problema também é resultado. Não inventamos tratamento. |
| `REFRIGERANTE LIMaO` (7.320 linhas) | **Padronizar** para `REFRIGERANTE LIMAO` (maiúsculas e sem espaços nas bordas, em 4 colunas de texto) | É a mesma coisa escrita de forma diferente; padronizar não muda o significado. |
| `#N/D` em `PR_CAT`/`PR_NOME` (3.650 linhas) | **Preencher com `NAO IDENTIFICADO`** e **manter as linhas** | É um valor ausente disfarçado. Preencher com um produto real seria inventar dado; remover apagaria itens de compras válidas (cliente, data e compra estão corretos). O rótulo deixa a ausência visível. |
| `DATA` como texto | **Converter para data** | Permite ordenar e filtrar por período corretamente. |
| Mesmo produto com 2 `PR_ID` | **Manter** | Pode ser marca, tamanho ou fornecedor diferente. Unificar sem confirmar com o cadastro alteraria dados possivelmente corretos. |
| 96.553 linhas idênticas | **Manter** (`remover_duplicatas=False`) | Duplicata **não confirmada**: sem coluna de quantidade, a repetição representa mais de uma unidade do mesmo produto na mesma compra. Remover subestimaria em 11,63% o volume de itens. |

In [15]:
# TRANSFORMAÇÃO: aplica a função de tratamento
df_limpo = tratar_base(df, remover_duplicatas=False)

# Comparação do diagnóstico ANTES x DEPOIS
comparacao = pd.DataFrame({
    "antes": diagnostico_qualidade(df),
    "depois": diagnostico_qualidade(df_limpo),
})
comparacao

,antes,depois
registros,830000,830000
colunas,14,10
colunas 100% vazias,4,0
valores ausentes (total),3320000,0
textos vazios/só espaços,0,0
linhas com '#N/D',3650,0
linhas com minúscula em PR_NOME,7320,0
nomes de produto distintos,118,118
linhas duplicadas,96553,96553
DATA é do tipo data?,False,True


In [16]:
# Conferências pontuais do tratamento
print("Produto padronizado:", df_limpo.loc[df_limpo["PR_ID"] == 69, "PR_NOME"].unique())
print("Linhas 'NAO IDENTIFICADO':", (df_limpo["PR_CAT"] == "NAO IDENTIFICADO").sum())
print("Tipos após o tratamento:")
print(df_limpo.dtypes)

Produto padronizado: <StringArray>
['REFRIGERANTE LIMAO']
Length: 1, dtype: string
Linhas 'NAO IDENTIFICADO': 3650
Tipos após o tratamento:
DATA         datetime64[us]
CO_ID                 int64
CL_ID                 int64
CL_GENERO            string
CL_EC                 int64
CL_FHL                int64
CL_SEG               string
PR_ID                 int64
PR_CAT               string
PR_NOME              string
dtype: object


**Antes × depois:** os registros continuam **830.000** (nenhum dado válido foi perdido); as colunas caíram de 14 para 10 (só as vazias saíram); os 3.320.000 "valores ausentes" eram todos das colunas vazias e foram a 0; `#N/D` e a grafia inconsistente foram a 0; os nomes de produto continuam 118 (`#N/D` virou `NAO IDENTIFICADO` e `LIMaO` virou `LIMAO`, sem juntar produtos diferentes); a `DATA` passou a ser do tipo data; as duplicatas continuam 96.553 por decisão justificada.

## Sprint 4 — Análise

### 4.1 Estatísticas básicas — número de filhos (`CL_FHL`)
**Coluna escolhida:** `CL_FHL` (número de filhos), a única coluna numérica com significado de negócio — as demais numéricas são identificadores (`CO_ID`, `CL_ID`, `PR_ID`) ou códigos de categoria (`CL_EC`), e média de identificador não significa nada.

**Unidade analisada: o cliente.** O número de filhos é um atributo do cliente que se repete em todos os itens que ele compra. Se calcularmos a média direto nas 830.000 linhas, clientes que compram mais itens "pesam" mais. Por isso, primeiro deixamos **uma linha por cliente**.

In [17]:
# Uma linha por cliente (atributos do cliente são constantes, conferido na Sprint 1)
clientes = df_limpo.drop_duplicates(subset="CL_ID")
print("Clientes:", len(clientes))

# Função de RESUMO aplicada à unidade correta
resumo_filhos = resumo_numerico(clientes["CL_FHL"])
resumo_filhos

Clientes: 1000


{'contagem_validos': 1000, 'media': 1.14, 'minimo': 0, 'maximo': 4}

In [18]:
# Comparação: a média calculada (errada) por linha/item
print("Média de filhos por LINHA (item):", round(df_limpo["CL_FHL"].mean(), 2))

# Distribuição dos clientes por número de filhos
dist = clientes["CL_FHL"].value_counts().sort_index()
pd.DataFrame({"clientes": dist, "percentual_%": (dist / dist.sum() * 100).round(1)})

Média de filhos por LINHA (item): 1.15


,clientes,percentual_%
CL_FHL,,
0,527,52.7
1,125,12.5
2,130,13.0
3,121,12.1
4,97,9.7


**Interpretação (unidade: cliente):**
1. **Média de 1,14 filho por cliente**, entre 1.000 clientes válidos (nenhum ausente), com **mínimo 0 e máximo 4**. A média é puxada para baixo porque **52,7% dos clientes (527) não têm filhos** — a mediana é 0.
2. **Média por item ≠ média por cliente:** calculada nas linhas, a média seria 1,15, porque clientes com filhos registram um pouco mais de itens. A diferença é pequena aqui, mas mostra por que a unidade precisa ser definida antes do cálculo.

In [19]:
# Bônus — a mesma função de resumo reaproveitada em outra medida: itens por compra (unidade: compra)
itens_por_compra = df_limpo.groupby("CO_ID").size()
resumo_numerico(itens_por_compra)

{'contagem_validos': 18471, 'media': 44.94, 'minimo': 1, 'maximo': 89}

Cada compra tem em média **44,94 itens** (mínimo 1, máximo 89), considerando as 18.471 compras.

### 4.2 Filtro — produtos infantis (categoria `ACESSORIOS`)
**Pergunta de negócio:** os produtos infantis (chupeta e mordedor, únicos itens de `ACESSORIOS`) são comprados principalmente por clientes com filhos?

In [20]:
# FILTRO: apenas itens da categoria ACESSORIOS
acessorios = df_limpo[df_limpo["PR_CAT"] == "ACESSORIOS"]
print("Itens de ACESSORIOS:", len(acessorios))
print("Produtos:", acessorios["PR_NOME"].unique())
print("Compras com acessórios:", acessorios["CO_ID"].nunique())
print("Clientes que compraram acessórios:", acessorios["CL_ID"].nunique())

# Participação dos clientes SEM filhos: nos acessórios x na base inteira
sem_filhos_acess = (acessorios["CL_FHL"] == 0).mean() * 100
sem_filhos_total = (df_limpo["CL_FHL"] == 0).mean() * 100
print(f"\n% dos itens de ACESSORIOS comprados por clientes sem filhos: {sem_filhos_acess:.1f}%")
print(f"% de TODOS os itens comprados por clientes sem filhos:       {sem_filhos_total:.1f}%")

Itens de ACESSORIOS: 14557
Produtos: <StringArray>
['MORDEDOR', 'CHUPETA']
Length: 2, dtype: string
Compras com acessórios: 9224
Clientes que compraram acessórios: 1000

% dos itens de ACESSORIOS comprados por clientes sem filhos: 52.5%
% de TODOS os itens comprados por clientes sem filhos:       52.5%


**O que o filtro mede:** itens (linhas) de chupeta e mordedor. O resultado chama atenção: clientes **sem filhos** respondem por praticamente a mesma fatia dos itens infantis que respondem do total de itens. Ou seja, **ter filhos não muda o consumo de produtos infantis nesta base** — o que pode indicar compras para terceiros (presentes, netos, sobrinhos), cadastro de filhos desatualizado ou dados gerados de forma simulada. Antes de usar o campo `CL_FHL` para campanhas, vale validar o cadastro.

### 4.3 Agrupamento 1 — itens por categoria
**O que mede:** quantidade de **itens** (linhas) por categoria de produto e a participação de cada uma. Não é número de clientes nem faturamento.

In [21]:
# GROUPBY 1: contagem de ITENS por categoria
itens_categoria = df_limpo.groupby("PR_CAT").size().sort_values(ascending=False)
pd.DataFrame({
    "itens": itens_categoria,
    "participacao_%": (itens_categoria / itens_categoria.sum() * 100).round(2),
})

,itens,participacao_%
PR_CAT,,
ALIMENTOS,434767,52.38
HIGIENE,155574,18.74
LIMPEZA,145754,17.56
BEBIDAS,43299,5.22
PET,32399,3.90
ACESSORIOS,14557,1.75
NAO IDENTIFICADO,3650,0.44


### 4.4 Agrupamento 2 — perfil por segmento de cliente (A, B, C)
**O que mede:** para cada segmento, quantos **clientes distintos**, quantas **compras distintas** e quantos **itens** existem, além de duas médias derivadas. Aqui as quatro unidades aparecem separadas de propósito, para não confundir.

In [22]:
# GROUPBY 2: clientes, compras e itens por segmento
por_segmento = df_limpo.groupby("CL_SEG").agg(
    clientes=("CL_ID", "nunique"),
    compras=("CO_ID", "nunique"),
    itens=("PR_ID", "size"),
)
por_segmento["compras_por_cliente"] = (por_segmento["compras"] / por_segmento["clientes"]).round(2)
por_segmento["itens_por_compra"] = (por_segmento["itens"] / por_segmento["compras"]).round(2)
por_segmento["%_clientes"] = (por_segmento["clientes"] / por_segmento["clientes"].sum() * 100).round(1)
por_segmento["%_itens"] = (por_segmento["itens"] / por_segmento["itens"].sum() * 100).round(1)
por_segmento

,clientes,compras,itens,compras_por_cliente,itens_por_compra,%_clientes,%_itens
CL_SEG,,,,,,,
A,84,1492,67736,17.76,45.40,8.4,8.2
B,645,11843,530163,18.36,44.77,64.5,63.9
C,271,5136,232101,18.95,45.19,27.1,28.0


**Interpretação:**
- **Agrupamento 1:** `ALIMENTOS` concentra mais da metade dos itens; `HIGIENE` e `LIMPEZA` vêm em seguida. `NAO IDENTIFICADO` (antigo `#N/D`) representa uma fração pequena, então a ausência não compromete a análise por categoria.
- **Agrupamento 2:** o segmento **B** é o maior em clientes, compras e itens. Como a participação de cada segmento nos itens é muito parecida com a participação nos clientes, e as médias de compras por cliente e de itens por compra são semelhantes entre A, B e C, **o segmento determina quantos clientes existem, mas não muda o comportamento de compra**.

## Sprint 5 — Exportação e comunicação

### 5.1 Gravação da base tratada

O arquivo `varejo_limpo.csv` gerado tem cerca de 43 MB. Como o envio de arquivos pelo site do GitHub aceita no máximo 25 MB, no repositório ele está compactado em `varejo_limpo.csv.zip` (o CSV dentro é exatamente o gerado por esta célula). Para gerar novamente, basta executar o notebook.

In [23]:
# GRAVAÇÃO: base tratada sem o índice do pandas
df_limpo.to_csv("varejo_limpo.csv", index=False)

# Conferência: relendo o arquivo gerado
conferencia = pd.read_csv("varejo_limpo.csv")
print("Registros ANTES (base original):", len(df), "| colunas:", df.shape[1])
print("Registros DEPOIS (varejo_limpo):", len(conferencia), "| colunas:", conferencia.shape[1])
print("Colunas exportadas:", list(conferencia.columns))

Registros ANTES (base original): 830000 | colunas: 14
Registros DEPOIS (varejo_limpo): 830000 | colunas: 10
Colunas exportadas: ['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME']


### 5.2 Resumo dos problemas tratados ou mantidos

| Problema | Quantidade | Tratamento |
|---|---|---|
| Colunas vazias (`Unnamed: 10` a `13`) | 4 colunas (100% nulas) | Removidas |
| Nulos nas colunas reais | 0 | Nada a tratar |
| Textos vazios / só espaços | 0 | Nada a tratar |
| Grafia inconsistente (`LIMaO`) | 7.320 linhas | Padronizado (maiúsculas) |
| `#N/D` (ausência disfarçada) | 3.650 linhas | Rótulo `NAO IDENTIFICADO`; linhas mantidas |
| `DATA` em texto | 830.000 linhas | Convertida para data |
| Produto com 2 códigos | 109 nomes | Mantido — exige confirmação no cadastro |
| Linhas idênticas | 96.553 linhas | Mantidas — representam unidades repetidas, não erro confirmado |
| **Registros** | **830.000 → 830.000** | Nenhum item válido descartado |

## Conclusões
1. **A base registra itens, não clientes:** 830.000 itens em 18.471 compras de 1.000 clientes, entre 04/01/2019 e 08/12/2022 — em média **44,9 itens por compra** e **18,5 compras por cliente**.
2. **Qualidade boa, com 2 problemas reais:** 3.650 itens com produto `#N/D` (0,44%) e 7.320 com grafia inconsistente (0,88%); nenhum nulo ou texto vazio nas colunas com dados.
3. **Alimentos dominam o volume:** a categoria `ALIMENTOS` responde por 52,4% dos itens; `HIGIENE` e `LIMPEZA` somam mais 36,3%.
4. **Segmento B é o maior público** (64,5% dos clientes e 63,9% dos itens), mas os três segmentos compram com frequência (17,8 a 19,0 compras por cliente) e volume (44,8 a 45,4 itens por compra) muito parecidos.
5. **52,7% dos clientes não têm filhos** (média de 1,14 filho por cliente) e, mesmo assim, ficam com 52,5% dos itens de chupeta e mordedor — exatamente a mesma fatia que têm do total de itens.

**Recomendação:** antes de segmentar campanhas por número de filhos, **validar o cadastro de clientes** (campo `CL_FHL`), já que ele não explica a compra de produtos infantis; em paralelo, corrigir na origem o cadastro do produto `PR_ID 107` (`#N/D`) e investigar os 109 produtos com dois códigos.

**Limitação:** a base **não tem valor (R$) nem quantidade** dos itens. Por isso não é possível falar em faturamento ou ticket médio, e não dá para confirmar se as 96.553 linhas idênticas são unidades repetidas ou erro de registro.

## Referências
- Material da disciplina: *Revisão orientada do mini-projeto — Módulo 1, Semana 07* (SCTEC).
- Documentação do pandas: `read_csv`, `isna`, `duplicated`, `groupby`, `to_datetime`, `to_csv` — https://pandas.pydata.org/docs/
- Base: *Base Varejo* — Kaggle, https://www.kaggle.com/datasets/namespaiva/base-varejo